In [115]:
from analysis.models.data import Data

with open("../data/data.json", "r") as f:
    data = Data.model_validate_json(f.read())

In [116]:
PROMPT = """
You will be given a description of an issue on GitHub. If you think a SWE AI agent will be able to fix it, respond with "yes". If you think a SWE AI agent will not be able to fix it, respond with "no".
"""

In [150]:
import litellm
import dotenv
from pydantic import BaseModel
from analysis.models.swe_bench import Instance

# load info from the dotenv file
values = dotenv.dotenv_values()

class QueryResponse(BaseModel):
    can_resolve: bool
    reasoning: str | None = None

class Response(BaseModel):
    instance_id: str
    problem_statement: str
    responses: list[QueryResponse]
    usages: list[litellm.Usage]

    @property
    def likelihood(self) -> float:
        """
        Calculate the likelihood of the AI SWE agent being able to resolve the issue.
        """
        if len(self.responses) == 0:
            return 0.0
        return sum([1 if r.can_resolve else 0 for r in self.responses]) / len(self.responses)
    
    @property
    def cost(self) -> float:
        """
        Calculate the cost of the response.
        """
        return self.input_cost + self.output_cost

    @property
    def input_cost(self) -> float:
        """
        Calculate the cost of the input.
        """
        result = 0.0
        for usage in self.usages:
            cost, _ = litellm.cost_calculator.cost_per_token(
                model="claude-3-7-sonnet-20250219",
                prompt_tokens=usage.prompt_tokens,
                completion_tokens=usage.completion_tokens,
                cache_creation_input_tokens=usage.cache_creation_input_tokens,
                cache_read_input_tokens=usage.cache_read_input_tokens,
            )
            result += cost
        return result
    
    @property
    def output_cost(self) -> float:
        """
        Calculate the cost of the output.
        """
        result = 0.0
        for usage in self.usages:
            _, cost = litellm.cost_calculator.cost_per_token(
                model="claude-3-7-sonnet-20250219",
                prompt_tokens=usage.prompt_tokens,
                completion_tokens=usage.completion_tokens,
                cache_creation_input_tokens=usage.cache_creation_input_tokens,
                cache_read_input_tokens=usage.cache_read_input_tokens,
            )
            result += cost
        return result

system_prompt = """
Evaluate if an AI SWE agent can correctly resolve GitHub issues. Consider technical failure modes where AI might:

1. Misunderstand complex system interactions or side effects
2. Fail to handle edge cases not explicitly mentioned
3. Miss subtle bugs in proposed solutions
4. Incorrectly interpret ambiguous technical requirements
5. Propose solutions that don't scale under load or for edge data
6. Generate code with subtle performance issues or memory leaks
7. Fail to consider backwards compatibility

Additionally, be especially skeptical of the AI's ability to handle:
8. Issues involving multiple interacting systems or components
9. Language-specific behaviors that vary between versions or implementations
10. Environment-specific issues (OS differences, container behavior, etc.)
11. Complex database and query optimization problems
12. Character encoding and internationalization edge cases
13. Issues requiring knowledge of implicit conventions not documented in code

However, an AI agent CAN likely handle:
1. Well-defined bugs with clear reproduction steps
2. Issues confined to a single component or subsystem
3. Problems with explicit error messages or symptoms
4. Framework-specific issues where the behavior is well-documented
5. Issues that have been clearly described with sufficient context

Return True if:
- The issue is well-defined and isolated to a specific component
- The problem description clearly identifies the unexpected behavior
- The required fix is likely confined to a small number of related files
- The issue doesn't require understanding complex interactions or side effects

Return False if:
- The issue spans multiple systems or components
- The problem requires understanding undocumented behaviors or conventions
- The issue involves complex performance, scaling, or optimization concerns
- The bug appears in specific environments or configurations only
- The description is ambiguous or lacks clear reproduction steps

When in doubt, consider whether a junior developer with basic knowledge of the framework could reasonably address the issue with documentation alone.
"""

system_prompt = """
Evaluate if an AI SWE agent can correctly resolve GitHub issues. AI agents often struggle with issues requiring deep technical understanding of system internals or subtle implementation details.

Return False if ANY of these conditions apply:
1. The issue involves database-specific behaviors or differences between database backends
2. The issue involves parsing, tokenization, or regular expressions
3. The issue involves configuration or settings handling with special characters or syntax
4. The issue requires understanding type coercion or handling across system boundaries
5. The issue involves mathematical or computational edge cases
6. The issue requires knowledge of implementation differences across platforms or versions
7. The issue involves undocumented assumptions or behaviors in the codebase
8. The issue appears in some environments but not others
9. The issue involves character encoding, internationalization, or locale-specific behavior
10. The issue requires understanding subtle interactions between components
11. The issue involves memory management, resource handling, or performance optimization
12. The issue involves parsing or generating complex data formats (JSON, XML, etc.)

Return True ONLY if ALL of these conditions are met:
1. The issue has a clear, isolated cause with explicit error messages
2. The solution likely involves adding a missing check or handling a clearly defined edge case
3. The issue can be reproduced consistently with the given steps
4. The fix is likely confined to a single function or small section of code
5. The issue doesn't involve multiple interacting components or systems
6. The code in question follows standard patterns documented in the framework
7. The solution doesn't require deep understanding of internal implementation details

When evaluating, assume that AI agents have knowledge of standard library APIs and common framework concepts, but struggle with:
- Implementation-specific quirks
- Internal framework mechanisms
- Cross-component interactions
- Environment-specific behaviors
- Undocumented conventions
- Performance implications
- Security considerations beyond obvious patterns

Remember: It's better to be cautious and classify an issue as requiring human expertise (False) than to overestimate AI capabilities (True). When in doubt, choose False.
"""

agent_system_prompt = """
You are OpenHands agent, a helpful AI assistant that can interact with a computer to solve tasks.

<ROLE>
Your primary role is to assist users by executing commands, modifying code, and solving technical problems effectively. You should be thorough, methodical, and prioritize quality over speed.
* If the user asks a question, like "why is X happening", don't try to fix the problem. Just give an answer to the question.
</ROLE>

<EFFICIENCY>
* Each action you take is somewhat expensive. Wherever possible, combine multiple actions into a single action, e.g. combine multiple bash commands into one, using sed and grep to edit/view multiple files at once.
* When exploring the codebase, use efficient tools like find, grep, and git commands with appropriate filters to minimize unnecessary operations.
</EFFICIENCY>

<FILE_SYSTEM_GUIDELINES>
* When a user provides a file path, do NOT assume it's relative to the current working directory. First explore the file system to locate the file before working on it.
* If asked to edit a file, edit the file directly, rather than creating a new file with a different filename.
* For global search-and-replace operations, consider using `sed` instead of opening file editors multiple times.
</FILE_SYSTEM_GUIDELINES>

<CODE_QUALITY>
* Write clean, efficient code with minimal comments. Avoid redundancy in comments: Do not repeat information that can be easily inferred from the code itself.
* When implementing solutions, focus on making the minimal changes needed to solve the problem.
* Before implementing any changes, first thoroughly understand the codebase through exploration.
* If you are adding a lot of code to a function or file, consider splitting the function or file into smaller pieces when appropriate.
</CODE_QUALITY>

<VERSION_CONTROL>
* When configuring git credentials, use "openhands" as the user.name and "openhands@all-hands.dev" as the user.email by default, unless explicitly instructed otherwise.
* Exercise caution with git operations. Do NOT make potentially dangerous changes (e.g., pushing to main, deleting repositories) unless explicitly asked to do so.
* When committing changes, use `git status` to see all modified files, and stage all files necessary for the commit. Use `git commit -a` whenever possible.
* Do NOT commit files that typically shouldn't go into version control (e.g., node_modules/, .env files, build directories, cache files, large binaries) unless explicitly instructed by the user.
* If unsure about committing certain files, check for the presence of .gitignore files or ask the user for clarification.
</VERSION_CONTROL>

<PULL_REQUESTS>
* When creating pull requests, create only ONE per session/issue unless explicitly instructed otherwise.
* When working with an existing PR, update it with new commits rather than creating additional PRs for the same issue.
* When updating a PR, preserve the original PR title and purpose, updating description only when necessary.
</PULL_REQUESTS>

<PROBLEM_SOLVING_WORKFLOW>
1. EXPLORATION: Thoroughly explore relevant files and understand the context before proposing solutions
2. ANALYSIS: Consider multiple approaches and select the most promising one
3. TESTING:
   * For bug fixes: Create tests to verify issues before implementing fixes
   * For new features: Consider test-driven development when appropriate
   * If the repository lacks testing infrastructure and implementing tests would require extensive setup, consult with the user before investing time in building testing infrastructure
4. IMPLEMENTATION: Make focused, minimal changes to address the problem
5. VERIFICATION: Test your implementation thoroughly, including edge cases
</PROBLEM_SOLVING_WORKFLOW>

<SECURITY>
* Only use GITHUB_TOKEN and other credentials in ways the user has explicitly requested and would expect.
* Use APIs to work with GitHub or other platforms, unless the user asks otherwise or your task requires browsing.
</SECURITY>

<ENVIRONMENT_SETUP>
* When user asks you to run an application, don't stop if the application is not installed. Instead, please install the application and run the command again.
* If you encounter missing dependencies:
  1. First, look around in the repository for existing dependency files (requirements.txt, pyproject.toml, package.json, Gemfile, etc.)
  2. If dependency files exist, use them to install all dependencies at once (e.g., `pip install -r requirements.txt`, `npm install`, etc.)
  3. Only install individual packages directly if no dependency files are found or if only specific packages are needed
* Similarly, if you encounter missing dependencies for essential tools requested by the user, install them when possible.
</ENVIRONMENT_SETUP>

<TROUBLESHOOTING>
* If you've made repeated attempts to solve a problem but tests still fail or the user reports it's still broken:
  1. Step back and reflect on 5-7 different possible sources of the problem
  2. Assess the likelihood of each possible cause
  3. Methodically address the most likely causes, starting with the highest probability
  4. Document your reasoning process
* When you run into any major issue while executing a plan from the user, please don't try to directly work around it. Instead, propose a new plan and confirm with the user before proceeding.
</TROUBLESHOOTING>
"""

agent_user_prompt = """
<uploaded_files>
/workspace/{workspace_dir_name}
</uploaded_files>
I've uploaded a python code repository in the directory {workspace_dir_name}. Consider the following issue description:

<issue_description>
{instance.problem_statement}
</issue_description>


Can you help me implement the necessary changes to the repository to test whether the issue in <issue_description> was resolved?
I will take care of all changes to any of the non-test files. This means you DON'T have to modify the actual logic and ONLY have to update test logic and tests!
Your task is to make the minimal changes to tests files in the /workspace directory to reproduce the issue in the <issue_description>, i.e., such that the generated tests fail in the current state (where the issue is unresolved) and pass when the issue will be resolved.
Follow these steps to reproduce the issue:
1. As a first step, it might be a good idea to explore the repo to familiarize yourself with its structure.
2. Create a script `reproduction.py` to reproduce the error and execute it with `python reproduction.py` using the BashTool, to confirm the error
3. Edit the sourcecode of the repo to integrate your reproduction script into the test framework
4. Run the test framework and make sure your tests fail! Only submit FAILING tests! Never submit passing tests.
{test_instructions}Your thinking should be thorough and so it's fine if it's very long.
"""

tools = [
    {
        "type": "function",
        "function": {
            "name": "record_github_issue_resolution",
            "description": "Record the results of an analysis on a GitHub issue to determine if an AI SWE agent can accurately resolve it without making critical technical errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "can_resolve": {
                        "type": "boolean",
                        "description": "Whether an AI SWE agent can correctly resolve this issue without making technical errors or misunderstanding the problem. (True/False)"
                    },
                    "reasoning": {
                        "type": "string",
                        "description": "Brief explanation of the reasoning (max 100 words)"
                    }
                },
                "required": ["can_resolve", "reasoning"]
            }
        }
    }
]

def query(instance: Instance, n: int = 10, temperature: float = 1.0) -> Response:
    
    responses = []
    usage = []

    for _ in range(n):
        response = litellm.completion(
            **values,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": f"AI SWE agent system prompt: {agent_system_prompt}"
                },
                {
                    "role": "user",
                    "content": f"AI SWE agent issue prompt: {agent_user_prompt}"
                },
                {
                    "role": "user",
                    "content": f"GitHub issue description: {instance.problem_statement}",
                    "cache_control": {"type": "ephemeral"}
                },
            ],
            temperature=temperature,
            tools=tools,
            tool_choice={
                "type": "function",
                "function": {"name": "record_github_issue_resolution"},
            },
        )
        args = response.choices[0].message.tool_calls[0].function.arguments
        responses.append(QueryResponse.model_validate_json(args))
        usage.append(response.usage)

    return Response(
        instance_id=instance.instance_id,
        problem_statement=instance.problem_statement,
        responses=responses,
        usages=usage,
    )

In [118]:
import random
split = random.sample(data.dataset.instances, k=50)

for instance in split:
    print("Processing instance:", instance.instance_id)
    response = query(instance)
    print(f"Resolution likelihood: {response.likelihood}")
    with open("../data/responses_w_prompts.json", "a") as f:
        f.write(response.model_dump_json() + "\n")



Processing instance: django__django-16595
Resolution likelihood: 0.2
Processing instance: scikit-learn__scikit-learn-14629
Resolution likelihood: 0.3
Processing instance: django__django-16454
Resolution likelihood: 0.1
Processing instance: django__django-16263
Resolution likelihood: 0.0
Processing instance: sphinx-doc__sphinx-7748
Resolution likelihood: 0.0
Processing instance: pytest-dev__pytest-10081
Resolution likelihood: 0.8
Processing instance: sphinx-doc__sphinx-8120
Resolution likelihood: 0.0
Processing instance: sympy__sympy-21930
Resolution likelihood: 0.6
Processing instance: django__django-14608
Resolution likelihood: 1.0
Processing instance: django__django-12708
Resolution likelihood: 0.1
Processing instance: django__django-15731
Resolution likelihood: 0.9
Processing instance: matplotlib__matplotlib-25332
Resolution likelihood: 0.0
Processing instance: sympy__sympy-21596
Resolution likelihood: 0.0
Processing instance: matplotlib__matplotlib-26291
Resolution likelihood: 0.2


In [119]:
responses = []
with open("../data/responses_w_prompts.json", "r") as f:
    for line in f:
        responses.append(Response.model_validate_json(line))

In [173]:
# Compute ROC curve against a particular system in the data
from analysis.models.swe_bench import Evaluation
from sklearn.metrics import roc_curve
import pandas as pd
import altair as alt

def roc(responses: list[Response], evaluation: Evaluation) -> alt.Chart:
    """
    Compute the ROC curve for a given system.
    """
    df = pd.DataFrame([{
        "instance_id": r.instance_id,
        "likelihood": r.likelihood,
        "resolved": evaluation.results.is_resolved(r.instance_id),
    } for r in responses])

    fpr, tpr, thresholds = roc_curve(df['resolved'], df['likelihood'])
    curve = pd.DataFrame({
        "fpr": fpr,
        "tpr": tpr,
        "threshold": thresholds
    })

    p_ratio = df['resolved'].sum() / len(df)
    curve["accuracy"] = p_ratio * curve["tpr"] + (1 - p_ratio) * (1 - curve["fpr"])

    chart = alt.Chart(curve).mark_line().encode(
        x=alt.X("fpr", title="False Positive Rate"),
        y=alt.Y("tpr", title="True Positive Rate"),
        tooltip=["fpr", "tpr", "threshold", "accuracy"],
    )

    return chart

roc(responses, data.systems["20250203_openhands_4x_scaled"])

alt.Chart(...)

In [174]:
from pathlib import Path
from functools import reduce

filepaths = [
    Path("../data/responses.json"),
    Path("../data/responses_w_temp.json"),
    Path("../data/responses_w_temp_2.json"),
    Path("../data/responses_w_temp_2_no_reasoning.json"),
    Path("../data/responses_w_prompts.json"),
]

multi_responses = {}

for filepath in filepaths:
    system_responses = []
    with filepath.open('r') as f:
        for line in f:
            system_responses.append(Response.model_validate_json(line))
    
    multi_responses[filepath.name] = system_responses

reduce(lambda x, y: x + y, [roc(responses, data.systems["20250203_openhands_4x_scaled"]) for responses in multi_responses.values()])

alt.LayerChart(...)

In [121]:
def poor_performers(responses: list[Response], evaluation: Evaluation):
    """Get the instance IDs where the predicted likelihood differs significantly from the actual resolution status."""
    df = pd.DataFrame([{
        "instance_id": r.instance_id,
        "likelihood": r.likelihood,
        "resolved": evaluation.results.is_resolved(r.instance_id),
        "problem_statement": r.problem_statement,
    } for r in responses])

    # Calculate the difference between predicted and actual resolution status
    df['diff'] = abs(df['likelihood'] - df['resolved'])

    # Sort by the difference
    df = df.sort_values(by='diff', ascending=False)

    return df.head(10)

poor_performers(responses, data.systems["20250203_openhands_4x_scaled"])

,instance_id,likelihood,resolved,problem_statement,diff
30,django__django-14672,0.0,True,Missing call `make_hashable` on `through_field...,1.0
33,django__django-7530,0.0,True,makemigrations router.allow_migrate() calls fo...,1.0
6,sphinx-doc__sphinx-8120,0.0,True,locale/<language>/LC_MESSAGES/sphinx.po transl...,1.0
36,django__django-15525,0.0,True,loaddata fails on non-default database when na...,1.0
11,matplotlib__matplotlib-25332,0.0,True,[Bug]: Unable to pickle figure with aligned la...,1.0
16,django__django-11211,0.0,True,Prefetch related is not working when used GFK ...,1.0
2,django__django-16454,0.1,True,Management command subparsers don’t retain err...,0.9
45,django__django-13028,0.1,True,Queryset raises NotSupportedError when RHS has...,0.9
19,sphinx-doc__sphinx-9698,0.1,True,An index entry with parens was registered for ...,0.9
9,django__django-12708,0.1,True,Migration crashes deleting an index_together i...,0.9


In [156]:
def reasons(responses: list[Response], instance_id: str) -> list[str]:
    """
    Get the reasons for the given instance ID.
    """
    for response in responses:
        if response.instance_id == instance_id:
            return [r.reasoning for r in response.responses]
    return []

reasons(responses, "django__django-14672")

["This issue involves undocumented internal behavior of Django's ORM system and requires understanding complex interactions between components. The bug is related to hashability of ManyToManyRel objects specifically when using through_fields with proxy models. It requires deep understanding of Django's internal implementation details around model relations, object comparison, and the make_hashable utility function that isn't obvious from documentation.",
 "This issue involves understanding Django's internal model relationship implementation, specifically ManyToManyRel's identity property. The bug occurs when hashing a list type field that needs to be made hashable. This requires deep knowledge of Django framework internals, model relationship implementation details, and the specific make_hashable function's behavior. An AI would struggle to identify where and how to modify the ManyToManyRel class without a complete understanding of Django's complex ORM architecture.",
 "This issue invo

In [154]:
from collections import Counter
import numpy as np

# Compute accuracy and cost for each system
rows = []

for system_id, system_responses in multi_responses.items():
    # Classification entropy
    counts = Counter(r.likelihood for r in system_responses)
    normalized_counts = [count / sum(counts.values()) for count in counts.values()]
    entropy = -sum(p * np.log2(p) if p > 0 else 1e-10 for p in normalized_counts)

    row = {
        "system": system_id,
        "cost": sum(r.cost for r in system_responses),
        "entropy": entropy
    }

    rows.append(row)

df = pd.DataFrame(rows)
df

,system,cost,entropy
0,responses.json,1.928359,0.937269
1,responses_w_temp.json,1.820188,1.400334
2,responses_w_temp_2.json,2.025594,2.640924
3,responses_w_temp_2_no_reasoning.json,3.260416,1.731472
4,responses_w_prompts.json,2.435563,2.978885


In [170]:

def multi_summary(multi_responses: dict[str, list[Response]], evaluation: Evaluation) -> pd.DataFrame:
    rows = []
    for system_id, system_responses in multi_responses.items():
        for response in system_responses:
            row = {
                "system": system_id,
                "instance_id": response.instance_id,
                "repo": response.instance_id.split("__")[0],
                "likelihood": response.likelihood,
                "resolved": evaluation.results.is_resolved(response.instance_id),
            }
            rows.append(row)
    return pd.DataFrame(rows)

df = multi_summary(multi_responses, data.systems["20250203_openhands_4x_scaled"]).groupby(["system", "repo"]).agg(
    {
        "likelihood": "mean",
        "resolved": "mean",
        "instance_id": "count"
    }
).reset_index().rename(columns={"instance_id": "count"})

df["diff"] = df["likelihood"] - df["resolved"]
summary_df = df.groupby(["system", "repo"]).agg(
    {
        "likelihood": "mean",
        "resolved": "mean",
        "diff": "mean",
        "count": "max"
    }
).reset_index()

summary_df[summary_df["repo"] == "django"]

,system,repo,likelihood,resolved,diff,count
1,responses.json,django,0.911538,0.615385,0.296154,26
9,responses_w_prompts.json,django,0.263636,0.590909,-0.327273,22
17,responses_w_temp.json,django,0.841667,0.750000,0.091667,24
27,responses_w_temp_2.json,django,0.391667,0.708333,-0.316667,24
38,responses_w_temp_2_no_reasoning.json,django,0.272000,0.760000,-0.488000,25
